In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "nolte2021targeted")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "chimpanzees_Paper_Helping&Coop_Nolte&Call_Data.csv")
complete_path_2 = os.path.join(original_data_pathway, "bonobos_Paper_Helping&Coop_Nolte&Call_Data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1, low_memory=False)
df2 = pd.read_csv(complete_path_2, low_memory=False)

In [3]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"closeproximity": "close_proximity",
        "appoperated":"app_operated",
        "tooltransfer":"tool_transfer",
        "startdate":"start_date",
        "receiver":"ape",
        "birthdayreceiver":"date_of_birth_receiver",
        "helper":"ape_2",
        "birthdayhelper":"date_of_birth_helper",
        "time":"time_in_seconds",
        "sessiontransferred":"session_transferred",
        "toolreceiver":"tool_receiver", 
        "toolhelper":"tool_helper"}, inplace=True)
    x['study_id']="nolte2021targeted"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

fulldf = fulldf.assign(role='receiver')
fulldf = fulldf.assign(role_2='helper')

In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)
    fulldf['ape_2'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

fulldf['dyad']=fulldf.ape.str.cat(fulldf.ape_2, sep='_')

In [5]:
fulldf[['month','day', 'year']] = fulldf['start_date'].str.split('/',expand=True)

In [6]:
spe_2=[]  
for index, row in fulldf.iterrows():
    if row['study']=="cooperation":
        spe_2.append("2")
    elif row['study'] in ["helping", "helping1","helping2"]:
        spe_2.append("1")
    else:
        spe_2.append("")
fulldf = fulldf.assign(experiment=spe_2)

In [7]:
# fulldf.columns
fulldf.rename(columns={"ape": "participant", "ape_2":"participant_2"}, inplace=True)

In [8]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
fulldf= fulldf.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    fulldf[x] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
    fulldf[x] = pd.to_datetime(fulldf[x])
    fulldf[y] = pd.to_datetime(fulldf[y])
    fulldf[k] = (fulldf[x] - fulldf[y]).dt.days//365

In [9]:

fulldf=fulldf[['study_id','experiment', 'year','month','day', 
               'participant','age_in_years','sex','role',
        'participant_2','age_in_years_2','sex_2','role_2',
          'species','dyad', 'condition', 'session', 'study', 'time_in_seconds',
       'banging', 'scratching', 'reaching', 'close_proximity', 'app_operated',
       'tool_transfer', 'session_transferred', 
       'tool_receiver', 'tool_helper' ]]


In [10]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'nolte2021targeted_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'nolte2021targeted_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

